In [1]:
pip install alerce
pip install astropy

SyntaxError: invalid syntax (2684364580.py, line 1)

In [2]:
import pandas as pd
from alerce.core import Alerce
from astropy.coordinates import SkyCoord
import astropy.units as u


In [3]:
input_csv = "DIAObjects_with_ForcedPhotometry_RADEC.csv"
output_csv = "Alerce_LSST_matches.csv"
radius_arcsec = 1.0  # search radius

df = pd.read_csv(input_csv)
coords = SkyCoord(df["ra"], df["dec"], unit="deg")

In [4]:
client = Alerce()

matches = []

all_classes = set()

for i, coord in enumerate(coords):
    print(f"Checking {i+1}/{len(coords)} RA={coord.ra.deg}, DEC={coord.dec.deg}")
    
    try:
        # Query ALeRCE for objects near this position
        result = client.query_objects(
            survey="lsst",
            ra=coord.ra.deg,
            dec=coord.dec.deg,
            radius=radius_arcsec,
            format="pandas"
        )
        
        if len(result) > 0:
            for _, row in result.iterrows():
                prob_dict = row.get("probabilities", {})
                all_classes.update(prob_dict.keys())
                
                if prob_dict:
                    max_class = max(prob_dict, key=prob_dict.get)
                    max_prob = prob_dict[max_class]
                else:
                    max_class = None
                    max_prob = None

                match = {
                    "input_ra": coord.ra.deg,
                    "input_dec": coord.dec.deg,
                    "alerce_oid": row["oid"],
                    "matched_ra": row["meanra"],
                    "matched_dec": row["meandec"],
                    "max_class": max_class,
                    "max_prob": max_prob,
                }

                
                for cls, val in prob_dict.items():
                    match[cls] = val

                matches.append(match)
            print("  → Found match: class =", max_class, "prob =", max_prob)
    
    except Exception as e:
        print("  Error:", e)

# Save matches to CSV
if matches:
    df_out = pd.DataFrame(matches)

    # Make sure all class columns exist
    for cls in all_classes:
        if cls not in df_out.columns:
            df_out[cls] = None

    df_out.to_csv(output_csv, index=False)
    print(f"\nSaved {len(df_out)} matches to {output_csv}")
else:
    print("\nNo matches found.")

Checking 1/289 RA=53.52887885543894, DEC=-28.59599123218368
Checking 2/289 RA=53.53114509579025, DEC=-28.583778873911413
Checking 3/289 RA=53.53796203950908, DEC=-28.580889088961708
Checking 4/289 RA=53.5802505228509, DEC=-28.51951030889495
Checking 5/289 RA=53.64795797710596, DEC=-28.4700977069736
  → Found match: class = None prob = None
Checking 6/289 RA=53.6836005329128, DEC=-28.435547512302307
  → Found match: class = None prob = None
Checking 7/289 RA=53.821447094433786, DEC=-28.29353420106637
Checking 8/289 RA=53.29038907053458, DEC=-28.757310898277595
  → Found match: class = None prob = None
Checking 9/289 RA=53.23883529678109, DEC=-28.738119632447813
Checking 10/289 RA=53.21623860561227, DEC=-28.715315616579804
Checking 11/289 RA=53.24348423937094, DEC=-28.692275856332852
Checking 12/289 RA=53.34090791603708, DEC=-28.69678673097237
Checking 13/289 RA=53.36510338608062, DEC=-28.6283805443682
Checking 14/289 RA=53.372525571300976, DEC=-28.593931075244782
Checking 15/289 RA=53.3